In [3]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
import matplotlib.pyplot as plt


In [4]:
# =====================================================
# SEO ADDER LOGICAL GATES
# =====================================================
def gate_A(qc, A):
    qc.h(A)
    qc.t(A)

def compute_and(qc, x, y, A):
    qc.cx(x, A)
    qc.cx(y, A)
    qc.cx(A, y)
    qc.cx(A, x)
    qc.tdg(x)
    qc.tdg(y)
    qc.t(A)
    qc.cx(A, y)
    qc.cx(A, x)
    qc.h(A)
    qc.s(A)

def uncompute_and(qc, x, y, A):
    qc.sdg(A)
    qc.h(A)
    qc.cx(A, y)
    qc.cx(A, x)
    qc.t(x)
    qc.t(y)
    qc.tdg(A)
    qc.cx(A, y)
    qc.cx(A, x)
    qc.cx(y, A)
    qc.cx(x, A)

# =====================================================
# BUILD SEO ADDER CIRCUIT DYNAMICALLY
# =====================================================
def build_seo_adder(A, B):
    qc = QuantumCircuit(12, 5)

    a_qubits = [0, 3, 6, 9]
    b_qubits = [1, 4, 7, 10]

    # ENCODE A
    for i in range(4):
        if (A >> i) & 1:
            qc.x(a_qubits[i])

    # ENCODE B
    for i in range(4):
        if (B >> i) & 1:
            qc.x(b_qubits[i])

    qc.barrier()

    # PREPARE ANCILLAS IN |A> STATE
    gate_A(qc, 2)
    gate_A(qc, 5)
    gate_A(qc, 8)
    gate_A(qc, 11)

    qc.barrier()

    # Forward operations (Compute)
    compute_and(qc, 0, 1, 2)
    qc.cx(2, 3)
    qc.cx(2, 4)

    compute_and(qc, 3, 4, 5)
    qc.cx(2, 5)
    qc.cx(5, 6)
    qc.cx(5, 7)

    compute_and(qc, 6, 7, 8)
    qc.cx(5, 8)
    qc.cx(8, 9)
    qc.cx(8, 10)

    compute_and(qc, 9, 10, 11)
    qc.cx(8, 11)

    # Reverse operations (Uncompute)
    qc.cx(8, 9)
    qc.cx(5, 8)
    uncompute_and(qc, 6, 7, 8)

    qc.cx(5, 6)
    qc.cx(2, 5)
    uncompute_and(qc, 3, 4, 5)

    qc.cx(2, 3)
    uncompute_and(qc, 0, 1, 2)

    qc.barrier()

    # Final sums
    qc.cx(0, 1)
    qc.cx(3, 4)
    qc.cx(6, 7)
    qc.cx(9, 10)
    
    qc.barrier()

    # Measurements
    qc.measure(1, 0)
    qc.measure(4, 1)
    qc.measure(7, 2)
    qc.measure(10, 3)
    qc.measure(11, 4)

    return qc

In [ ]:
# ====================================================
# DEPOLARIZING NOISE MODEL
# ====================================================
error_1 = depolarizing_error(0.005, 1)
error_2 = depolarizing_error(0.01, 2)

noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(error_1, ['x', 'h', 's', 'sdg', 't', 'tdg'])
noise_model.add_all_qubit_quantum_error(error_2, ['cx'])

sim = AerSimulator(noise_model=noise_model)

# ====================================================
# RUN ALL 256 COMBINATIONS
# ====================================================
shots = 100
results = []

print("Running 256 test cases. Please wait...\n")

for a in range(16):
    for b in range(16):
        qc = build_seo_adder(a, b)
        compiled = transpile(qc, sim)
        counts = sim.run(compiled, shots=shots).result().get_counts()

        correct = a + b
        correct_output = format(correct, '05b')  # 5-bit binary

        correct_counts = counts.get(correct_output, 0)
        ER = 1 - (correct_counts / shots)

        total_ED = 0
        total_relative_ED = 0
        for output, freq in counts.items():
            noisy_decimal = int(output, 2)
            ED = abs(correct - noisy_decimal)
            total_ED += ED * freq
            
            if correct != 0:
                total_relative_ED += (ED / correct) * freq

        mean_ED = total_ED / shots
        D = 30  # Max possible output for 4-bit addition is 30
        NMED = mean_ED / D
        MRED = total_relative_ED / shots

        results.append((a, b, correct_output, counts, ER, NMED, MRED))

# ====================================================
# CALCULATE AGGREGATE METRICS
# ====================================================
sum_er = sum(item[4] for item in results)
sum_nmed = sum(item[5] for item in results)
sum_mred = sum(item[6] for item in results)

er = sum_er / 256
nmed = sum_nmed / 256
mred = sum_mred / 256

# ====================================================
# DISPLAY RESULTS
# ====================================================
print("===================================")
print("SEO ADDER (256 Inputs)")
print("Depolarizing Noise")
print("===================================")
print(f"ER(%) : {er * 100:.4f}")
print(f"NMED  : {nmed:.6f}")
print(f"MRED  : {mred:.6f}")

er_array = [result[4] for result in results]
# print("\nER Array:")
# print(er_array)

Running 256 test cases. Please wait...

SEO ADDER (256 Inputs)
Depolarizing Noise
ER(%) : 37.3594
NMED  : 0.068878
MRED  : 0.207019

ER Array:
[0.45999999999999996, 0.37, 0.39, 0.4, 0.36, 0.36, 0.27, 0.38, 0.31000000000000005, 0.31999999999999995, 0.32999999999999996, 0.44999999999999996, 0.36, 0.36, 0.35, 0.31000000000000005, 0.32999999999999996, 0.35, 0.47, 0.45999999999999996, 0.39, 0.28, 0.4, 0.35, 0.35, 0.37, 0.37, 0.37, 0.39, 0.41000000000000003, 0.31999999999999995, 0.38, 0.37, 0.35, 0.39, 0.39, 0.38, 0.39, 0.19999999999999996, 0.33999999999999997, 0.41000000000000003, 0.30000000000000004, 0.33999999999999997, 0.43000000000000005, 0.4, 0.37, 0.38, 0.43000000000000005, 0.31000000000000005, 0.45999999999999996, 0.31999999999999995, 0.37, 0.48, 0.36, 0.37, 0.4, 0.31000000000000005, 0.39, 0.43000000000000005, 0.30000000000000004, 0.4, 0.43000000000000005, 0.33999999999999997, 0.31999999999999995, 0.31999999999999995, 0.37, 0.43000000000000005, 0.38, 0.4, 0.38, 0.4, 0.4, 0.37, 0.3399